In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
def merge_delta_produto(df, tabela_destino):
    """
    Realiza MERGE (upsert) de um DataFrame em uma tabela Delta Lake de configurações.

    Regras de chave:
    ----------------
    - Chave de negócio: (cliente, produto)
    - Mantém sempre APENAS 1 linha por (cliente, produto) na tabela destino.
    - Caso o DataFrame contenha mais de um registro para o mesmo (cliente, produto),
      será mantida a linha com a data_execucao mais recente.

    Parâmetros:
    -----------
    df : DataFrame
        DataFrame com os dados a serem inseridos/atualizados.
        Deve conter, obrigatoriamente:
            - cliente
            - produto
            - data_execucao
        Os demais campos podem ser nulos.
        
    tabela_destino : str
        Nome da tabela Delta Lake (ex: "workspace.bronze_etl.config_clientes").

    Retorno:
    --------
    None
    """

    # ⚠️ Verifica se há dados
    if df is None or df.limit(1).count() == 0:
        print("⚠️ Nenhum dado para atualizar.")
        return

    # 🔁 Deduplica por (cliente, produto) mantendo somente a última execução
    janela = Window.partitionBy("cliente", "produto").orderBy(F.col("data_execucao").desc())

    df_dedup = (
        df
        .withColumn("_rn", F.row_number().over(janela))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    total = df_dedup.count()
    print(f"✅ Total de registros (deduplicados) a carregar: {total}")

    # 🔍 Verifica se a tabela Delta já existe
    if spark.catalog.tableExists(tabela_destino):
        print(f"📦 Tabela {tabela_destino} já existe — atualizando dados...")

        delta_table = DeltaTable.forName(spark, tabela_destino)

        # 🔑 Condição de MERGE pela chave de negócio
        condicao_merge = "t.cliente = s.cliente AND t.produto = s.produto"
        print("🔑 Usando chaves: cliente, produto")

        (
            delta_table.alias("t")
            .merge(
                df_dedup.alias("s"),
                condicao_merge
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print(f"✅ MERGE concluído com sucesso em {tabela_destino}")

    else:
        print(f"🆕 Tabela {tabela_destino} não existe — criando nova tabela...")

        (
            df_dedup.write
            .format("delta")
            .partitionBy("cliente")  # opcional
            .mode("append")
            .option("overwriteSchema", "true")
            .saveAsTable(tabela_destino)
        )

        print(f"✅ Tabela {tabela_destino} criada e dados inseridos com sucesso.")

In [0]:
def merge_config(df, tabela_destino: str):
    """
    Realiza MERGE da tabela de configuração de clientes (contabilidade)
    usando como chaves: cliente + mes_referencia.

    Parâmetros:
    - df_config: DataFrame Spark com as colunas padronizadas
    - tabela_destino: tabela Delta no formato catalogo.schema.tabela
    """

    # 🚨 Validação mínima
    required_cols = {"cliente", "mes_referencia"}
    if not required_cols.issubset(df.columns):
        raise ValueError(
            f"❌ O DataFrame precisa conter as colunas: {required_cols}. "
            f"Colunas recebidas: {df.columns}"
        )

    if spark.catalog.tableExists(tabela_destino):
        print(f"🔄 Tabela {tabela_destino} encontrada — realizando MERGE...")

        delta_tbl = DeltaTable.forName(spark, tabela_destino)

        (
            delta_tbl.alias("t")
            .merge(
                df.alias("s"),
                """
                t.cliente = s.cliente
                AND t.mes_referencia = s.mes_referencia
                """
            )
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute()
        )

        print("✅ MERGE concluído com sucesso.")
    else:
        print(f"🆕 Criando tabela {tabela_destino} pela primeira vez...")

        df.write \
                .format("delta") \
                .partitionBy("mes_referencia") \
                .mode("append") \
                .option("overwriteSchema", "true") \
                .saveAsTable(tabela_destino)

        print(" Tabela criada com sucesso.")